In [ ]:
from idc_index import IDCClient

client = IDCClient()
client.fetch_index("sm_index")  # slide microscopy attributes, incl. specimen tissue type
client.get_idc_version()

In [ ]:
collections = ["cptac_luad", "cptac_lscc"]
df = client.sql_query(f"""
SELECT
    s.ContainerIdentifier,
    i.PatientID,
    i.collection_id,
    s.primaryAnatomicStructure_CodeMeaning AS primary_site,
    i.SeriesInstanceUID,
    i.series_aws_url,
    i.series_size_MB
FROM index AS i
JOIN sm_index AS s USING (SeriesInstanceUID)
WHERE i.collection_id IN ({", ".join(f"'{c}'" for c in collections)})
    -- Neoplasm, Primary; the rest are normal or unlabeled
    AND s.primaryAnatomicStructureModifier_code_designator_value_str = 'SCT:86049000'
    -- hematoxylin and eosin
    AND list_has_all(s.staining_usingSubstance_code_designator_value_str, ['SCT:12710003', 'SCT:36879007'])
ORDER BY s.ContainerIdentifier
""")

In [ ]:
df

In [ ]:
df.rename(
    columns={
        "ContainerIdentifier": "slide",
        "PatientID": "case_id",
        "SeriesInstanceUID": "series_uid",
    }
)[["slide", "case_id", "collection_id", "primary_site", "series_uid"]].to_csv(
    "../metadata/cptac-nsclc-primary-tumor-slide-metadata.csv", index=False
)

In [ ]:
with open("../manifests/cptac-nsclc-primary-tumor-slide.s5cmd", "w") as f:
    f.write(f"# IDC {client.get_idc_version()}\n")
    for url in df["series_aws_url"]:
        f.write(f"cp {url} .\n")